# NOVA RETAIL — SQL Forense

## Consultas operativas de investigación

Este notebook contiene las consultas SQL reales que se ejecutarían directamente sobre la base de datos transaccional para detectar los patrones de riesgo identificados en el análisis.

Estas no son consultas de reporting. Son consultas forenses diseñadas para:
- Identificar anomalías que no aparecen en los reportes estándar
- Priorizar nodos de riesgo
- Generar evidencia defendible para escalamiento

---

### Setup

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../data/processed/nova_retail.db")

def run_query(sql):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

---

## Query 01: Ranking de Tiendas por Discrepancia Acumulada

Esta es la consulta base que ordena las tiendas por severidad total de pérdida.

In [2]:
sql = """
SELECT 
    store_id,
    store_name,
    route,
    COUNT(*) as total_receptions,
    ROUND(AVG(discrepancy_pct), 2) as avg_discrepancy,
    ROUND(SUM(abs(discrepancy_value)), 2) as total_value_at_risk
FROM reception_analysis
GROUP BY store_id, store_name, route
ORDER BY total_value_at_risk DESC
LIMIT 15
"""

run_query(sql)

---

## Query 02: Patrón 05:00 AM

Consulta que identifica todas las recepciones fuera del horario operativo normal.

In [3]:
sql = """
SELECT 
    strftime('%H', reception_time) as hour,
    route,
    COUNT(*) as count,
    ROUND(AVG(discrepancy_pct), 2) as avg_discrepancy
FROM reception_analysis
GROUP BY hour, route
HAVING hour < '07' OR hour > '20'
ORDER BY count DESC
"""

run_query(sql)

---

## Query 03: Ghost SKUs con movimiento activo

Identifica SKUs que existen en SAP pero no tienen precio registrado en AS400, y han sido movidos en los últimos 90 días.

In [4]:
sql = """
SELECT 
    p.sku,
    p.description,
    p.sap_price,
    p.as400_price,
    COUNT(*) as movements_last_90d
FROM products p
JOIN reception_analysis ra ON p.sku = ra.sku
WHERE 
    p.as400_price IS NULL 
    OR p.as400_price = 0
GROUP BY p.sku, p.description, p.sap_price, p.as400_price
ORDER BY movements_last_90d DESC
LIMIT 20
"""

run_query(sql)

---

## Query 04: Score de Riesgo Compuesto

Esta es la consulta que genera la Matriz de Riesgo Unificada, combinando todos los vectores en un solo score.

In [5]:
sql = """
WITH store_metrics AS (
    SELECT 
        store_id,
        AVG(discrepancy_pct) as avg_disc,
        SUM(CASE WHEN strftime('%H', reception_time) < '07' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) as pct_off_hours,
        COUNT(DISTINCT CASE WHEN as400_price IS NULL THEN sku END) as ghost_skus
    FROM reception_analysis
    GROUP BY store_id
)
SELECT 
    s.store_id,
    s.store_name,
    s.route,
    ROUND(avg_disc, 2) as avg_discrepancy,
    ROUND(pct_off_hours * 100, 1) as pct_off_hours,
    ghost_skus,
    ROUND(
        (avg_disc * 0.5) + 
        (pct_off_hours * 100 * 0.3) + 
        (ghost_skus * 0.2),
        2
    ) as risk_score
FROM store_metrics sm
JOIN stores s ON sm.store_id = s.store_id
ORDER BY risk_score DESC
LIMIT 15
"""

run_query(sql)

---

### Nota Metodológica

Todas estas consultas están diseñadas para ser ejecutadas directamente sobre el sistema transaccional sin necesidad de exportar datos a Excel. Esta es la forma en la que realmente se trabaja en operaciones y prevención de pérdidas.